# Capstone — Google Search Ranking & Discoverability

**Lane:** Content Refresh Priority

This notebook builds the reproducible analysis and the static research page. It automatically uses the approved gated warehouse when a local Hugging Face token is available and falls back to the bundled anonymized starter slice otherwise. The executed receipt records which source was used.

## 1. Question

Which content pages look worth reviewing first when editorial time is limited? The output is a ranked reviewer queue. A strategist acts on it by checking the live page, intent, demand, seasonality, and business context before choosing a refresh action.

In [1]:
from pathlib import Path
import sys
import pandas as pd

repo_candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/FlyRank-ML'), Path('/content/flyrank-ml')]
repo_root = next((p for p in repo_candidates if (p / 'work' / 'ml_track.py').exists()), Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from work.ml_track import (
    TARGET,
    ensure_dirs,
    load_analysis_frame,
    make_feature_matrix,
    run_artifacts,
    run_validation,
    write_json,
    write_paper_page,
)

ensure_dirs()
frame = load_analysis_frame()
print(f"Loaded {len(frame):,} rows across {frame['client_id'].nunique():,} client groups")
print(f"Observed snapshot-proxy base rate: {frame[TARGET].mean():.3f}")

artifacts = run_artifacts()
validation = artifacts['validation']
queue = artifacts['queue']
print('Question framed: rank pages for human refresh review.')

C:\Khalil\FlyRank-ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 111,133 rows across 49 client groups
Observed snapshot-proxy base rate: 0.644


Question framed: rank pages for human refresh review.


## 2. Data

The executed receipt records the source, row count, client-group count, target definition, and leakage exclusions. The model uses only pre-label-window fields; pseudonymous IDs are used for grouping and tracing, never as features. No names, domains, URLs, titles, raw queries, credentials, or raw warehouse exports are included in the public artifacts.

In [2]:
print(f"Rows: {validation['rows']:,}")
print(f"Client groups: {validation['clients']:,}")
print(f"Target definition: {validation['target_definition']}")
print(f"Target base rate: {frame[TARGET].mean():.3f}")
print(f"Excluded from features: {validation['leakage_excluded']}")

Rows: 111,133
Client groups: 49
Target definition: last-30d impressions < 80% of prior-30d impressions
Target base rate: 0.644
Excluded from features: ['trend_pct', 'trend_direction', 'content_id', 'client_id']


## 3. Methodology

Numeric performance/content fields are median-imputed with missingness flags; categorical context is one-hot encoded. The transparent baseline ranks staleness, visibility, CTR risk, and position risk. The model is a seeded random forest. The primary validation is a client-grouped holdout because the same pseudonymous client repeats across rows; a random row holdout is reported as a comparison. The leakage audit explicitly excludes the label-derived trend fields.

In [3]:
grouped = validation['grouped_split']
random_result = validation['random_split']
print(f"Grouped train/test rows: {grouped['train_rows']:,} / {grouped['test_rows']:,}")
print(f"Grouped train/test clients: {grouped['train_clients']:,} / {grouped['test_clients']:,}")
print(f"Feature count: {grouped['feature_count']}")
print('Seed:', 42)

Grouped train/test rows: 94,021 / 17,112
Grouped train/test clients: 39 / 10
Feature count: 76
Seed: 42


## 4. Results (vs baseline)

The grouped client holdout is the primary estimate. Precision@K is shown beside the base rate so a reader can distinguish ranking skill from the class prevalence.

In [4]:
result_table = pd.DataFrame([
    {'split': 'Random row holdout', 'approach': 'Baseline', **random_result['baseline']},
    {'split': 'Random row holdout', 'approach': 'Model', **random_result['model']},
    {'split': 'Grouped client holdout', 'approach': 'Baseline', **grouped['baseline']},
    {'split': 'Grouped client holdout', 'approach': 'Model', **grouped['model']},
])
display(result_table[['split', 'approach', 'base_rate', 'precision_at_20', 'precision_at_50', 'average_precision', 'roc_auc']].round(3))

,split,approach,base_rate,precision_at_20,precision_at_50,average_precision,roc_auc
0,Random row holdout,Baseline,0.644,0.90,0.82,0.724,0.613
1,Random row holdout,Model,0.644,1.00,1.00,0.847,0.770
2,Grouped client holdout,Baseline,0.598,0.65,0.58,0.710,0.651
3,Grouped client holdout,Model,0.598,0.50,0.58,0.716,0.696


## 5. Limitations

These results are observed associations in an anonymized starter slice. They do not prove that refreshing a page causes traffic recovery, do not predict or reverse-engineer Google's algorithm, and do not establish performance on future time windows or the gated warehouse. Missing history, seasonality, tracking changes, site migrations, low-volume volatility, and unobserved editorial decisions can change the queue.

In [5]:
limitations = [
    'snapshot-proxy label rather than future treatment outcome',
    f"data source: {validation['source']}",
    'observational data with possible selection and seasonality effects',
    'human review remains mandatory before any content action',
]
print('\n'.join(f'- {item}' for item in limitations))

- snapshot-proxy label rather than future treatment outcome
- data source: gated FlyRank warehouse release v20260703
- observational data with possible selection and seasonality effects
- human review remains mandatory before any content action


## 6. Ranked recommendations

Start with high-confidence pages that combine model risk and visible demand. Then choose a specific review path: CTR/snippet review, engagement/readability review, content expansion, or monitoring. The queue should organize editorial attention; it should not publish changes automatically.

In [6]:
display(queue[['rank', 'final_score', 'confidence', 'suggested_action', 'reason_codes']].head(15))
print('Action counts:', artifacts['queue_summary']['action_counts'])

,rank,final_score,confidence,suggested_action,reason_codes
0,1,89.108353,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...
1,2,88.415869,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...
2,3,88.047314,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...
3,4,87.972523,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...
4,5,87.913951,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...
5,6,87.902440,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...
6,7,87.851004,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...
7,8,87.633200,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...
8,9,87.483053,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...
9,10,87.464652,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...


Action counts: {'monitor': 53418, 'refresh': 48539, 'refresh_and_review_ctr': 5299, 'refresh_and_review_engagement': 3852, 'expand_and_refresh': 25}


## 7. Artifacts the paper embeds

The pipeline creates model-vs-baseline, action-mix, and feature-importance charts. It also writes compact JSON receipts that let a reviewer trace the paper numbers back to a fresh run.

In [7]:
paper_path = write_paper_page(artifacts)
print(f'Wrote paper: {paper_path}')
print('Wrote figures under work/figures/ and docs/assets/')
print('Wrote receipts under work/outputs/')

Wrote paper: C:\Khalil\FlyRank-ML\docs\index.html
Wrote figures under work/figures/ and docs/assets/
Wrote receipts under work/outputs/


## ML-12 — 5-minute demo outline + shareable cuts

**Demo outline:** (1) show the editorial decision and why a ranked queue is useful; (2) show the data contract and the excluded label-derived fields; (3) show the transparent baseline; (4) show the grouped holdout comparison; (5) show one feature-importance chart and one queue example; (6) close with limitations and the human-review rule.

**Short social post:** I built a leakage-aware content-refresh ranking workflow on FlyRank's anonymized data. Instead of treating a model score as a Google ranking oracle, I compared it with a transparent baseline under a client-grouped holdout and turned the output into reason-coded editorial actions. The paper and reproducible notebooks show what the data supports—and where it stops.

**Employer-facing summary:** I built a reproducible ML workflow that ranks content pages for human refresh review using anonymized search-performance data. I compared a seeded random forest with a transparent staleness/visibility baseline under a client-grouped holdout, including missingness flags and leakage checks. The result is a public-safe research page, executable notebooks, charts, and a reason-coded action queue rather than an unsupported causal claim.

In [8]:
required_sections = [
    '<h2>Abstract</h2>', '<h2>1. Introduction / problem</h2>', '<h2>2. Data</h2>',
    '<h2>3. Methodology</h2>', '<h2>4. Results</h2>', '<h2>5. Limitations & honest framing</h2>',
    '<h2>6. Ranked recommendations</h2>', '<h2>7. Reproducibility</h2>', '<h2>8. Acknowledgments & data credit</h2>',
]
html = paper_path.read_text(encoding='utf-8')
missing_sections = [section for section in required_sections if section not in html]
assert not missing_sections, missing_sections
assert 'https://flyrank.ai' in html
print('Paper section check passed: all 9 required sections and FlyRank data credit are present.')

Paper section check passed: all 9 required sections and FlyRank data credit are present.


## Self-check

- [x] Question, data, method, results, limitations, recommendations, reproducibility, and credit are present
- [x] The grouped holdout and base rate are visible
- [x] No label-derived fields or identifiers are model features
- [x] ML-12 is included in the closing cells
- [x] The generated page is ready for GitHub Pages after the repo owner enables `/docs`